# pdf-OCR-to-txt

Convert scanned, image-based PDF documents into plain text (`.txt`) files using 300 DPI rasterization (`poppler`) and `Tesseract-OCR`.

### Workflow:
1. **Run Step 1 once** per session to install the required Linux binaries and Python packages.
2. **Run Step 2 as many times as needed** to upload PDFs and automatically download timestamped `.txt` transcriptions without waiting for dependencies to reinstall.

## Step 1: Install Dependencies (Run Once Per Session)

In [ ]:
# 00_install_dependencies.py
import subprocess
import sys

def install_dependencies():
    print("Step 1/3: Updating apt package lists...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    
    print("Step 2/3: Installing system binaries (poppler-utils, tesseract-ocr)...")
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "poppler-utils", "tesseract-ocr"],
        check=True
    )
    
    print("Step 3/3: Installing Python libraries (pdf2image, pytesseract)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pdf2image", "pytesseract"],
        check=True
    )
    
    print("\nSetup complete! Dependencies are installed for this session.")
    print("You can now run Step 2 below as many times as needed.")

if __name__ == "__main__":
    install_dependencies()

## Step 2: Upload PDF(s) and Convert to Text (Re-run Anytime)

In [ ]:
# 01_upload_and_convert.py
import os
from datetime import datetime
from google.colab import files
from pdf2image import convert_from_path
import pytesseract

def process_pdf_to_txt():
    print("Please select your PDF file(s) to upload:")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Execution stopped.")
        return

    for filename in uploaded.keys():
        if not filename.lower().endswith('.pdf'):
            print(f"Skipping '{filename}': Not a PDF file.")
            continue

        print(f"\nConverting '{filename}' to high-resolution images (300 DPI)...")
        images = convert_from_path(filename, dpi=300)

        full_text = []
        total_pages = len(images)

        for i, image in enumerate(images):
            print(f"Running OCR on page {i + 1} of {total_pages}...")
            # Default Tesseract configuration (--psm 3).
            # For uniform single-column blocks, pass: config='--psm 6'
            # For multi-column or rotated pages with OSD, pass: config='--psm 1'
            page_text = pytesseract.image_to_string(image)
            full_text.append(page_text)

        final_text = "\n\n--- Page Break ---\n\n".join(full_text)

        base_name = os.path.splitext(filename)[0]
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_filename = f"{base_name}_ocr_{timestamp}.txt"

        with open(output_filename, "w", encoding="utf-8") as text_file:
            text_file.write(final_text)

        print(f"OCR complete for '{filename}'. Triggering download for '{output_filename}'...")
        files.download(output_filename)

if __name__ == "__main__":
    process_pdf_to_txt()